In [1]:
import sys
from pathlib import Path
from uncertainties import ufloat
import country_converter as coco
import pandas as pd
import math

# ------------------------ Run from Repo Root ------------------------
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

# ------------------------ File Paths ------------------------
CR_Box_Countries = BASE_DIR / "data" / "CR_Box_Countries_MS.csv"
country_list_csv = BASE_DIR / "data" / "STANDARD_COUNTRY_LIST.csv"

from country_pkg import Country

In [2]:
# ------------------------ Functions ------------------------

def generate_countries_from_iso_csv(country_csv_path, cr_box_csv_path=None):
    """
    Generate a dictionary of Country objects from a CSV with ISO-3 codes and country names.
    The country title (name) will be the ISO-3 code.
    Optionally load CR_Box properties from a separate CSV.
    If a country is not present in the CR_Box CSV, its properties are set to 0.
    """
    # Load the main country CSV
    df = pd.read_csv(country_csv_path, encoding='cp1252')
    required_cols = ['ISO-3', 'Country Name']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"CSV must have a column named '{col}'")
    
    # Load CR Box CSV if provided
    cr_box_df = None
    if cr_box_csv_path:
        cr_box_df = pd.read_csv(cr_box_csv_path, encoding='cp1252')
        if "Country" not in cr_box_df.columns:
            raise ValueError("CR Box CSV must have a 'Country' column")
        # Standardize the country names in CR Box CSV
        cr_box_df["Country"] = cr_box_df["Country"].apply(Country._cc.convert, to="name_short")
    
    countries = {}
    for _, row in df.iterrows():
        iso_code = row['ISO-3']
        country_name = row['Country Name']
        
        # Create country object using ISO code as its name
        c = Country(name=iso_code)
        c.properties['Country Name'] = country_name
        c.properties['ISO-3'] = iso_code
        
        # Load CR Box properties
        if cr_box_df is not None:
            standardized_name = Country._cc.convert(country_name, to="name_short")
            cr_row = cr_box_df[cr_box_df["Country"] == standardized_name]
            if not cr_row.empty:
                c.properties.update(cr_row.iloc[0].to_dict())
            else:
                # If country not in CR Box CSV, set numeric properties to 0
                for col in cr_box_df.columns:
                    if col != "Country":
                        c.properties[col] = 0
        
        countries[iso_code] = c
    
    return countries

In [3]:
## ------------------------ CR Box Reference ------------------------ 

Ind_Market_Rev_Per_MERV = {
    '17-20' : 2208.7e6,
    '5-8'   : 563.4e6,
    '9-12'  : 1271.8e6,
    '1-4'   : 171.7e6,
    '13-16' : 1878.1e6
}

Tot_Ind_Air_Filter = Ind_Market_Rev_Per_MERV['17-20']+Ind_Market_Rev_Per_MERV['5-8']+Ind_Market_Rev_Per_MERV['1-4']+Ind_Market_Rev_Per_MERV['9-12']+Ind_Market_Rev_Per_MERV['13-16']

Tot_Air_Filter = 20.8303e9

All_Market_Rev_Per_MERV = {
    '17-20' : Ind_Market_Rev_Per_MERV['17-20']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '5-8'   : Ind_Market_Rev_Per_MERV['5-8']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '9-12'  : Ind_Market_Rev_Per_MERV['9-12']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '1-4'   : Ind_Market_Rev_Per_MERV['1-4']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '13-16' : Ind_Market_Rev_Per_MERV['13-16']*Tot_Air_Filter/Tot_Ind_Air_Filter,
}

Price_Per_Filter = {
    '1-4'   : ufloat(1031.59,   107.26/2),
    '5-8'   : ufloat(1133.85,   447.48/2),
    '9-12'  : ufloat(1302.51,   554.53/2),
    '13-16' : ufloat(1951.25,   593.63/2),
    '17-20' : ufloat(22925.29,  3740.81/2)
}

Volume_to_Sale = 0.508*0.508*0.0254

Sales = {
    '1-4'   : All_Market_Rev_Per_MERV['1-4']/(Price_Per_Filter['1-4']*Volume_to_Sale),
    '5-8'   : All_Market_Rev_Per_MERV['5-8']/(Price_Per_Filter['5-8']*Volume_to_Sale),
    '9-12'  : All_Market_Rev_Per_MERV['9-12']/(Price_Per_Filter['9-12']*Volume_to_Sale),
    '13-16' : All_Market_Rev_Per_MERV['13-16']/(Price_Per_Filter['13-16']*Volume_to_Sale),
    '17-20' : All_Market_Rev_Per_MERV['17-20']/(Price_Per_Filter['17-20']*Volume_to_Sale)
}

Panel_Filter = ufloat(0.35,     0.35*0.1/2)
Scale_Up_Factor = 1/0.7

Usable_Filters = (Sales['13-16']+Sales['17-20']) * Panel_Filter * Scale_Up_Factor

In [4]:
## ------------------------ Countries ------------------------ 

## CHANGE

if "countries_dict" not in globals():
    countries_dict = generate_countries_from_iso_csv(country_list_csv, CR_Box_Countries)

sum_scale = 0
for country in countries_dict.values():
    msa = country.properties.get("MSA",0)
    mva = country.properties.get("MVA", 0)
    if msa == 1:
        country.properties["Big_6"] = True
        sum_scale += mva
    else:
        country.properties["Big_6"] = False

scale = Usable_Filters/sum_scale
for country in countries_dict.values():
    msa = country.properties.get("MSA",0)
    mva = country.properties.get("MVA", 0)
    x = scale * msa * mva
    if x.nominal_value < 50000:
        x = 0
    else:
        x = ufloat(math.floor(x.nominal_value / 4), x.std_dev)
    country.properties["CR Box"] = x
    country.properties["CADR CR Box"] = country.properties.get("CR Box", 0) * 126.13

s = 0
for country in countries_dict.values():
    x = country.properties.get("CR Box",0)
    s += x
    print(country.summary())

--- Aruba ---
Country Name: Aruba
ISO-3: ABW
MFS: 0
MSA: 0
MVA: 0
Relative MVA: 0
Big_6: False
CR Box: 0
CADR CR Box: 0.0
None
--- Afghanistan ---
Country Name: Afghanistan
ISO-3: AFG
MFS: 0
MSA: 0
MVA: 0
Relative MVA: 0
Big_6: False
CR Box: 0
CADR CR Box: 0.0
None
--- Angola ---
Country Name: Angola
ISO-3: AGO
MFS: 0
MSA: 0
MVA: 0
Relative MVA: 0
Big_6: False
CR Box: 0
CADR CR Box: 0.0
None
--- Albania ---
Country Name: Albania
ISO-3: ALB
MFS: 0
MSA: 0
MVA: 0
Relative MVA: 0
Big_6: False
CR Box: 0
CADR CR Box: 0.0
None
--- Andorra ---
Country Name: Andorra
ISO-3: AND
Country: Andorra
MFS: 80.85
MSA: 0.8317901235
MVA: 128595277.9
Relative MVA: 7.391312853e-06
Big_6: False
CR Box: 0
CADR CR Box: 0.0
None
--- United Arab Emirates ---
Country Name: United Arab Emirates
ISO-3: ARE
Country: United Arab Emirates
MFS: 72.5
MSA: 0.7458847737
MVA: 55761508810.0
Relative MVA: 0.003205022482
Big_6: False
CR Box: (2.5+/-1.5)e+05
CADR CR Box: (3.1+/-1.8)e+07
None
--- Argentina ---
Country Name: Arg